# 12. Preparación de base para determinantes del ingreso

Este notebook prepara una base analítica **interpretable y reproducible** para estudiar asociaciones entre características personales, laborales, del hogar y territoriales e ingreso laboral/de negocio en ENIGH 2024.

Alcance de esta etapa:

- Unidad de análisis: persona.
- Año: 2024.
- Universo: personas adultas (`edad >= 18`) con `ingreso_persona_laboral_negocio_tri > 0`.
- Target: `ingreso_persona_laboral_negocio_tri`, en montos nominales trimestrales.
- Fuente activa: `data/interim/revision_4/mart_persona_2018_2024.csv.gz`.
- No se entrenan modelos, no se crean particiones y no se usan deflactores, variables reales ni JKn.

Las bases con microdatos se escriben en `data/processed/determinantes_2024/` y quedan fuera de Git. Las tablas agregadas, figuras y documentación sí se versionan.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:,.4f}".format)

try:
    from IPython.display import display
except Exception:
    display = None


def mostrar(df, n=20):
    vista = df.head(n)
    if display is not None:
        display(vista)
    else:
        print(vista.to_string(index=False))


TABLE_DIR = PROJECT_ROOT / "reports" / "tables" / "preparacion_determinantes"
FIG_DIR = PROJECT_ROOT / "reports" / "figures" / "preparacion_determinantes"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "determinantes_2024"

print(PROJECT_ROOT)

C:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling


## Ejecución reproducible

La celda siguiente reconstruye todos los artefactos de la etapa 12 desde el mart nominal aprobado de `revision_4`. Si se vuelve a correr el notebook, las tablas, figuras y matrices se regeneran con la misma lógica.

In [2]:
from src.features.preparacion_determinantes import build_outputs

manifest = build_outputs(PROJECT_ROOT, execution_context="notebook_12_ejecutado_proceso_python_limpio_sin_nbclient")

print("Estado global:", manifest["validacion"]["estado_global"])
print("Notebook ejecutado:", manifest["validacion"]["notebook_ejecutado"])
print("Dimensiones:", manifest["dimensiones"])

Estado global: ok
Notebook ejecutado: True
Dimensiones: {'fuente_filas': 1203231, 'universo_filas': 141579, 'hogares_unicos': 80872, 'X_columnas': 67, 'continuas': 6, 'categoricas': 11}


## Universo analítico

Indicador del universo:

$$
\mathbb{1}(U_i) =
\mathbb{1}(\text{anio}_i = 2024)\,
\mathbb{1}(\text{edad}_i \ge 18)\,
\mathbb{1}(y_i > 0)
$$

donde \(y_i\) es `ingreso_persona_laboral_negocio_tri`.

Esta definición implica que los resultados futuros describen asociaciones **entre adultos con ingreso laboral/de negocio positivo**. No explican quién entra al universo de ingreso positivo.

In [3]:
flujo = pd.read_csv(TABLE_DIR / "flujo_universo.csv")
mostrar(flujo)

              paso                criterio  filas_antes  excluidas  filas_despues  hogares_unicos_despues
0        anio_2024            anio == 2024      1203231     894633         308598                   91414
1      edad_valida    edad valida y finita       308598          0         308598                   91414
2          adultos              edad >= 18       308598      89770         218828                   91389
3    target_valido  target valido y finito       218828          0         218828                   91389
4  target_positivo              target > 0       218828      77249         141579                   80872


## Target

El target se conserva en escala original nominal trimestral. El logaritmo se calcula solo como diagnóstico visual:

$$
\log(y_i)
$$

No reemplaza al target activo.

In [4]:
target_no_pond = pd.read_csv(TABLE_DIR / "target_resumen_no_ponderado.csv")
target_pond = pd.read_csv(TABLE_DIR / "target_resumen_ponderado.csv")
target_grupos = pd.read_csv(TABLE_DIR / "target_por_region_sexo_escolaridad.csv")

mostrar(target_no_pond)
mostrar(target_pond)

        metrica       n       media     mediana    desv_std    min      p01        p05        p10         p25         p50         p75         p90         p95  \
0  no_ponderado  141579 31,171.8697 24,245.8900 64,171.3549 2.9300 295.0800 1,516.3000 3,815.2100 12,433.4400 24,245.8900 38,225.2700 59,016.3800 79,239.1300   

           p99             max  
0 154,663.0300 17,021,739.1200  
                        metrica  n_muestral     suma_factor  media_ponderada      p01        p05        p10         p25  mediana_ponderada         p75  \
0  ponderado_factor_descriptivo      141579 61,105,085.0000      33,133.2329 342.3900 1,770.4809 4,402.1700 13,694.1243        25,081.9500 39,836.0400   

          p90         p95          p99             max  
0 62,853.2400 87,540.9600 169,472.5488 17,021,739.1200  


![Distribución del target](../reports/figures/preparacion_determinantes/target_hist_original_log.png)

### Contrastes descriptivos por grupos

Estas tablas usan el target original. Se incluyen medias/medianas muestrales y contrastes descriptivos ponderados con `factor`.

In [5]:
mostrar(target_grupos.sort_values(["variable", "mediana"], ascending=[True, False]), n=30)

           variable                                categoria      n       media     mediana     desv_std  media_ponderada_factor  mediana_ponderada_factor
6   nivelaprob_desc                                Doctorado    417 89,350.4683 73,858.6800  65,831.2188             95,111.4207               77,499.3207
10  nivelaprob_desc                                 Maestría   2243 76,960.8519 59,530.5900  71,511.1552             83,637.3380               62,379.8830
7   nivelaprob_desc                             Especialidad    360 85,087.2348 59,016.3900  84,824.3390             91,917.2629               62,567.2616
12  nivelaprob_desc                                   Normal    422 39,691.4817 37,863.7550  31,398.6148             41,450.9490               38,152.1700
9   nivelaprob_desc  Licenciatura o Ingeniería (profesional)  27905 47,317.7310 36,684.7800 119,744.6855             50,515.5577               37,907.5900
8   nivelaprob_desc          Estudios técnicos o comerciales   3513 33

![Target por región](../reports/figures/preparacion_determinantes/target_log_por_region.png)

![Target por sexo](../reports/figures/preparacion_determinantes/target_log_por_sexo.png)

![Target por escolaridad](../reports/figures/preparacion_determinantes/target_log_por_escolaridad.png)

## Predictores iniciales

La selección inicial es parsimoniosa. `factor`, `factor_hogar`, `est_dis` y `upm` se conservan como metadata, no como predictores. `est_socio`, entidad y tamaño de empresa principal quedan como diagnóstico/pendiente.

In [6]:
predictores = pd.read_csv(TABLE_DIR / "predictores_iniciales.csv")
faltantes = pd.read_csv(TABLE_DIR / "faltantes_predictores.csv")
diccionario = pd.read_csv(TABLE_DIR / "diccionario_variables.csv")

mostrar(predictores, n=40)
mostrar(faltantes, n=40)

                   variable        tipo              papel                                     transformacion                  referencia_o_tratamiento
0                      edad    continua  predictor inicial  sin escalar; estandarizada solo en matriz diag...                                       NaN
1                n_trabajos    continua  predictor inicial  sin escalar; estandarizada solo en matriz diag...                                       NaN
2      horas_trabajos_total    continua  predictor inicial  sin escalar; estandarizada solo en matriz diag...                                       NaN
3                 tot_integ    continua  predictor inicial  sin escalar; estandarizada solo en matriz diag...                                       NaN
4                   menores    continua  predictor inicial  sin escalar; estandarizada solo en matriz diag...                                       NaN
5                    p65mas    continua  predictor inicial  sin escalar; estandarizada s

## Codificación one-hot

Para cada variable categórica se usa one-hot con \(k-1\) columnas y una referencia explícita:

$$
D_{ic} = \mathbb{1}(x_i = c), \quad c \ne c_{\mathrm{ref}}
$$

No se guarda intercepto en la matriz; se deja para una etapa futura de modelado.

In [7]:
referencias = pd.read_csv(TABLE_DIR / "referencias_ohe.csv")
mapping_ohe = pd.read_csv(TABLE_DIR / "mapping_ohe.csv")

mostrar(referencias, n=30)
print("Categorias documentadas en mapping:", len(mapping_ohe))
print("Dummies activas:", int(mapping_ohe["dummy"].fillna("").ne("").sum()))

                   variable                                referencia                                           criterio  n_referencia
0                 sexo_desc                                    Hombre  categoria interpretable definida antes de modelar         82304
1           nivelaprob_desc                                   Ninguno  categoria interpretable definida antes de modelar          4145
2            region_banxico                                    Centro  categoria interpretable definida antes de modelar         35628
3              tam_loc_desc  Localidades con 100 000 y más habitantes  categoria interpretable definida antes de modelar         54544
4           parentesco_desc                                   Jefe(a)  categoria interpretable definida antes de modelar         68035
5             hablaind_desc                                        No  categoria interpretable definida antes de modelar        130582
6               segsoc_desc                            

## Estandarización diagnóstica

La matriz sin escalar es la referencia principal. Además se genera una versión diagnóstica donde solo las continuas se transforman como:

$$
z_i = \frac{x_i - \bar{x}}{s_x}
$$

Estos parámetros se calculan sobre la muestra completa solo para diagnóstico. En una etapa futura, si se crean train/test, el escalamiento debe ajustarse únicamente con entrenamiento.

In [8]:
params = pd.read_csv(TABLE_DIR / "parametros_estandarizacion.csv")
mostrar(params)

               variable   media  sd_muestral_ddof1  constante                                      formula
0                  edad 40.9757            14.3748      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1
1            n_trabajos  1.0555             0.3549      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1
2  horas_trabajos_total 43.7648            19.8207      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1
3             tot_integ  4.0133             1.8925      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1
4               menores  0.6973             0.9592      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1
5                p65mas  0.2597             0.5613      False  z_i = (x_i - media_x) / s_x; s_x con ddof=1


## Asociaciones exploratorias con el target

Continuas:

$$
\rho_s = \mathrm{corr}(R(x), R(y))
$$

con rangos y manejo estándar de empates vía `scipy`.

Dummies:

$$
r_{pb} = \mathrm{corr}(D, y)
$$

El punto biserial se calcula como Pearson entre un indicador 0/1 y el ingreso. Cada dummy compara una categoría contra el resto de categorías. Estas asociaciones son marginales, no ponderadas, sin p-values y no implican causalidad.

In [9]:
cont_assoc = pd.read_csv(TABLE_DIR / "correlaciones_continuas_spearman.csv")
bin_assoc = pd.read_csv(TABLE_DIR / "correlaciones_dummies_punto_biserial.csv")

mostrar(
    cont_assoc.assign(abs_rho=cont_assoc["spearman_ingreso_original"].abs())
    .sort_values("abs_rho", ascending=False)
    .drop(columns="abs_rho"),
    n=20,
)
mostrar(
    bin_assoc.assign(abs_r=bin_assoc["punto_biserial_ingreso_original"].abs())
    .sort_values("abs_r", ascending=False)
    .drop(columns="abs_r"),
    n=20,
)

               variable      tipo  n_valido  spearman_ingreso_original                                               nota
2  horas_trabajos_total  continua    141579                     0.3434  Correlacion de rangos exploratoria no ponderad...
5                p65mas  continua    141579                    -0.1394  Correlacion de rangos exploratoria no ponderad...
1            n_trabajos  continua    141579                     0.0853  Correlacion de rangos exploratoria no ponderad...
3             tot_integ  continua    141579                    -0.0534  Correlacion de rangos exploratoria no ponderad...
0                  edad  continua    141579                    -0.0446  Correlacion de rangos exploratoria no ponderad...
4               menores  continua    141579                    -0.0076  Correlacion de rangos exploratoria no ponderad...
                                                dummy        variable_original                                 categoria_vs_resto  n_valido  frecu

## Dependencia entre predictores

El diagnóstico revisa constantes, duplicados, correlaciones entre continuas, asociación entre categóricas, rango de matriz y VIF:

$$
VIF_j = \frac{1}{1 - R_j^2}
$$

El VIF depende de la codificación y es un diagnóstico de dependencia, no una regla automática de eliminación. Las asociaciones marginales anteriores tampoco son asociaciones condicionales de un modelo.

In [10]:
rango = pd.read_csv(TABLE_DIR / "dependencia_rango_matriz.csv")
constantes = pd.read_csv(TABLE_DIR / "dependencia_constantes.csv")
duplicadas = pd.read_csv(TABLE_DIR / "dependencia_duplicadas.csv")
vif = pd.read_csv(TABLE_DIR / "dependencia_vif.csv")
cramers = pd.read_csv(TABLE_DIR / "dependencia_categoricas_cramers_v.csv")

mostrar(rango)
print("Constantes:", len(constantes))
print("Duplicadas:", len(duplicadas))
mostrar(vif.sort_values("vif", ascending=False), n=20)
mostrar(cramers.sort_values("cramers_v", ascending=False), n=15)

    filas  columnas_X  columnas_con_intercepto  rango_con_intercepto  deficiencia_rango          estado
0  141579          67                       68                    68                  0  rango_completo
Constantes: 0
Duplicadas: 0
                                              columna     vif estado                                               nota
0   contrato_principal_desc__no_aplica_sin_contrat... 36.4987     ok  VIF_j = 1/(1-R_j^2). En dummies depende de la ...
1                            subor_principal_desc__si 36.3891     ok  VIF_j = 1/(1-R_j^2). En dummies depende de la ...
2                         nivelaprob_desc__secundaria 12.5976     ok  VIF_j = 1/(1-R_j^2). En dummies depende de la ...
3        nivelaprob_desc__preparatoria_o_bachillerato 11.8357     ok  VIF_j = 1/(1-R_j^2). En dummies depende de la ...
4   nivelaprob_desc__licenciatura_o_ingenieria_pro... 11.7787     ok  VIF_j = 1/(1-R_j^2). En dummies depende de la ...
5                           nivelaprob_desc_

## Validaciones y salidas

Las validaciones comprueban universo, llave única, dimensiones consistentes entre `X`, `y` y metadata, ausencia de target/metadata en `X`, OHE con referencias, valores finitos y equivalencia del punto biserial con Pearson en un ejemplo pequeño.

In [11]:
import json

validaciones = pd.read_csv(TABLE_DIR / "validaciones_preparacion.csv")
with open(TABLE_DIR / "manifest_preparacion_determinantes.json", encoding="utf-8") as f:
    manifest_archivo = json.load(f)

mostrar(validaciones, n=30)
print("Estado manifest:", manifest_archivo["validacion"])
print("Directorio de bases fuera de Git:", PROCESSED_DIR)
print("Directorio de tablas versionadas:", TABLE_DIR)
print("Directorio de figuras versionadas:", FIG_DIR)

                                  validacion resultado                                            detalle  valor_scipy  valor_pearson  diferencia_abs  \
0                  universo_anio_edad_target        ok  Todas las filas finales cumplen anio 2024, eda...          NaN            NaN             NaN   
1                        llave_persona_unica        ok  Duplicados ['anio', 'folioviv', 'foliohog', 'n...          NaN            NaN             NaN   
2                         filas_X_y_metadata        ok  Dimensiones: X=(141579, 67), X_scaled=(141579,...          NaN            NaN             NaN   
3         sin_target_derivados_metadata_en_X        ok                  Columnas prohibidas en X: ninguna          NaN            NaN             NaN   
4               ohe_referencias_consistentes        ok  Cada variable categorica tiene exactamente una...          NaN            NaN             NaN   
5                            X_sin_infinitos        ok                        Valo

## Decisiones pendientes antes de modelar

- Aprobar estrategia inferencial.
- Definir partición futura considerando hogares.
- Ajustar codificador y escalador solo con entrenamiento si se crean particiones.
- Decidir si `est_socio` entra como predictor o permanece como diagnóstico.
- Decidir si `tam_emp_principal_desc` se recodifica o permanece fuera por redundancia estructural.
- Revisar variables pendientes con etiquetas ambiguas antes de ampliar `X`.